# Experiment 003: 3B SFT-to-GRPO run, seed 1 (Colab)

The 3b grpo result (.716 overall, hard tier .520, the one place rl still beats
supervised) rests on single seed. After 0.5b showed a 19pt seed range this is
the most important open question in the study: does the 3b endpoint replicate?

Frozen protocol v1, a100, condition sft_grpo, seed 1. Base evals are shared
with seed 0 and skip automatically, so this session runs sft, sft eval, grpo
and grpo eval, ~6.5h. Everything resume-safe, finished stages skip.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [ ]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 404.61   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '003_colab_3b_seed1', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260720_101151 | cu_before = 404.61
Mounted at /content/drive
closed previous session: 20260720_095236
saved open session: 20260720_101151
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__)


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-3dda6478-4d18-128b-3cc6-c161862f58b3)
cuda: True | torch: 2.11.0+cu128


## 2. Clone or update the repo (automatic)

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1463, done.
remote: Counting objects: 100% (720/720), done.
remote: Compressing objects: 100% (470/470), done.
remote: Total 1463 (delta 414), reused 547 (delta 247), pack-reused 743 (from 1)
Receiving objects: 100% (1463/1463), 43.97 MiB | 13.98 MiB/s, done.
Resolving deltas: 100% (833/833), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file the code expects.
If the secret isn't set, it falls back to a manual prompt.


In [ ]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


KeyboardInterrupt: Interrupted by user

## 5. Check the Drive record

In [ ]:
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')


## 6. Build the data splits

In [ ]:
!python scripts/build_data.py


## 7. Wiring check

In [ ]:
# !python scripts/run.py --config configs/003_colab_full_b3_seed0.yaml  # seed 0, ran 13 jul, kept for the record
!python scripts/run.py --config configs/003_colab_full_b3_seed1.yaml

## 8. Full run

**If colab disconnects:** re-run this cell, finished stages skip.

In [18]:
# seed 0 ran 13 jul with the line below, kept commented for the record
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/003_colab_full_b3_seed0.yaml --stage all
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/003_colab_full_b3_seed1.yaml --stage all

Streaming output truncated to the last 5000 lines.
│ │ step inside   │ we try to     │               │              │           │ │
│ │ <think></thi… │ factor -128   │               │              │           │ │
│ │ tags, then    │ into two      │               │              │           │ │
│ │ give only the │ numbers that  │               │              │           │ │
│ │ letter of the │ add up to -6. │               │              │           │ │
│ │ correct       │ The only way  │               │              │           │ │
│ │ option (A, B, │ to do this is │               │              │           │ │
│ │ C or D)       │ with 6 and    │               │              │           │ │
│ │ inside        │ -6, so we     │               │              │           │ │
│ │ <answer></an… │ have:         │               │              │           │ │
│ │ tags.         │               │               │              │           │ │
│ │               │ $$\sqrt[3]{-… │               │       

## 9. Peek at the results

Both seeds side by side so the seed question gets a first answer right here.

In [19]:
import json, os
D = '/content/drive/MyDrive/inm363'
for s in (0, 1):
    d = os.path.join(D, 'results', f'003_COLAB_FULL_B3_sft_grpo_seed{s}')
    sm = os.path.join(d, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"seed{s}: sft {j['sft_acc']}  grpo {j['grpo_acc']}")
    else:
        print(f'seed{s}: no summary yet ({d})')

seed0: sft {'easy': 0.608646188850967, 'medium': 0.4876847290640394, 'hard': 0.4393939393939394}  grpo {'easy': 0.8213879408418657, 'medium': 0.6912972085385879, 'hard': 0.5202020202020202}
seed1: sft {'easy': 0.5870307167235495, 'medium': 0.44499178981937604, 'hard': 0.43686868686868685}  grpo {'easy': 0.8373151308304891, 'medium': 0.6814449917898193, 'hard': 0.5505050505050505}


## 10. Close the units record

In [20]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 350.18    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
